In [1]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/model'
os.chdir(workdir)
os.getcwd()

In [2]:
import copy
import importlib

In [37]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr

import vcfparser as vcfp
import bndconvert as bndc

importlib.reload(vcfp)
importlib.reload(bndc)

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
# We need at least 1 path to population reference and 1 path to a callset
ref_path = 'data/gnomad.v4.1.sv.sites.vcf.gz'
callset_path = 'data/test_gridss_output.NA12878.vcf'

In [66]:
ref_df, ref_meta = vcfp.parse_vcf(ref_path, output_meta=True, line_limit=0, skip_format=True, info_limit_to=['END'])

In [67]:
ref_df = ref_df[(ref_df.ALT == '<DEL>') & (ref_df.FILTER_PASS == 1)]

In [35]:
callset_df, callset_meta = vcfp.parse_vcf(callset_path, output_meta=True)

In [42]:
callset_df_test = bndc.bnd_to_event(callset_df, strict_matching=False)
callset_df_test

In [43]:
callset_df_test

In [59]:
callset_df_test.loc[:, 'length'] = abs(callset_df_test['INFO_END'] - callset_df_test['POS'] + 1)
callset_df_test.loc[:, 'seq'] = callset_df_test['LALT'].str.extract('([ATGCNatgcn]+)')[0]

callset_df = callset_df_test[
    (
        (
            callset_df_test['LALT'].str.contains('^[ATGCNatgcn]+\\[', ) & \
            callset_df_test['RALT'].str.contains('\\][ATGCNatgcn]$', )
        ) | \
        (
            callset_df_test['RALT'].str.contains('^[ATGCNatgcn]+\\[', ) & \
            callset_df_test['LALT'].str.contains('\\][ATGCNatgcn]$', )
        )
    ) & \
    (callset_df_test['seq'].str.len() <= callset_df_test['length'])
]

In [60]:
def get_meta_missing_in_df(vcf_df, meta_dict):
    ids_for_removal = []
    for meta_id in vcfp.combine_meta_dict(meta_dict).keys():
        try:
            vcf_df[meta_id]
        except KeyError:
            ids_for_removal.append(meta_id)
    return ids_for_removal

def trim_meta(vcf_df, meta_dict):
    meta_local = vcfp.combine_meta_dict(meta_dict)
    missing_meta = get_meta_missing_in_df(vcf_df, meta_dict)
    print(f'Removing items {missing_meta} from meta since they are missing in the vcf_df')
    for item in missing_meta:
        del meta_local[item]
    meta_dict = vcfp.split_meta_dict(meta_local)
    return vcf_df, meta_dict

In [61]:
callset_df, callset_meta = trim_meta(callset_df, callset_meta)

In [68]:
vcfp.delete_mult_val_cols(ref_df, ref_meta)
vcfp.convert_col_types(ref_df, ref_meta)

vcfp.delete_mult_val_cols(callset_df, callset_meta)
vcfp.convert_col_types(callset_df, callset_meta)

In [63]:
# ref_df_sample = ref_df.sample(n=50000).sort_values(by=['CHROM', 'INFO_END', 'POS'])
# ref_meta_sample = copy.deepcopy(ref_meta)
# callset_df_sample = callset_df.sample(n=10000).sort_values(by=['CHROM', 'POS', 'INFO_END'])
# callset_meta_sample = copy.deepcopy(callset_meta)

In [69]:
ref_df_range = ref_df.rename(columns={'CHROM': 'Chromosome', 'POS': 'Start', 'INFO_END': 'End'})
ref_df_range = pr.PyRanges(ref_df_range)

In [70]:
callset_df_range = callset_df.rename(columns={'CHROM': 'Chromosome', 'POS': 'Start', 'INFO_END': 'End'})
callset_df_range = pr.PyRanges(callset_df_range)

In [71]:
overlaps = ref_df_range.intersect(callset_df_range)

In [72]:
overlapping = callset_df_range.overlap(ref_df_range)

In [73]:
overlapped_df_range = callset_df_range.join(ref_df_range, how='left', report_overlap=True)

In [74]:
overlapped_df_range

In [75]:
overlapped_df_range = overlapped_df_range.as_df()

In [76]:
overlapped_df_range.loc[:, 'overlap_frac'] = \
    (overlapped_df_range['Overlap'] * 2 / \
    (- overlapped_df_range['Start'] + overlapped_df_range['End'] - \
    overlapped_df_range['Start_b'] + overlapped_df_range['End_b'])).clip(lower=0)
# overlapped_df_range.loc[:, 'overlap_callset_only'] = \
#     overlapped_df_range['Overlap'] / \
#     (- overlapped_df_range['Start'] + overlapped_df_range['End'])

In [77]:
overlapped_df_range = overlapped_df_range.sort_values('overlap_frac', ascending=False).drop_duplicates(['ID'], keep='first').sort_index()

In [78]:
overlapped_df_range.loc[:, 'ref_matched'] = np.where(overlapped_df_range.overlap_frac >= 0.8, True, False)

In [79]:
overlapped_df_range.loc[:, 'length'] = overlapped_df_range['End'] - overlapped_df_range['Start']

In [80]:
overlapped_df_final = overlapped_df_range.drop(columns = [
    'Chromosome', 'Start', 'LALT', 'RALT', 'FORMAT_GT',
    'INFO_EVENT', 'INFO_SVTYPE', 'End', 'CHR2',
    'Start_b', 'ID_b', 'REF_b', 'ALT_b', 'End_b', 'QUAL_b', 'Overlap', 'overlap_frac'
])
overlapped_df_final = overlapped_df_final.set_index('ID')

In [81]:
overlapped_df_final

In [82]:
overlapped_df_final.to_csv('saved/gridss_overlaps.tsv', sep='\t')